# Redes Neuronales con PyTorch para Regresión (California Housing)

Vamos a comparar el rendimiento de nuestros modelos de redes neuronales con un modelo RandomForest entrenado previamente que logró un RMSE de aproximadamente 41.000 dólares.

## Configuración Inicial

In [ ]:
import numpy as npimport torchimport torch.nn as nnimport torch.optim as optimfrom torch.utils.data import TensorDataset, DataLoaderfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import root_mean_squared_errorimport os

In [ ]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
# Determinar el dispositivo (GPU si está disponible, de lo contrario CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed(RANDOM_STATE) # Inicializar semilla para GPU
else:
    device = torch.device("cpu")
    print(f"\nUsando dispositivo: {device}")
num_workers = os.cpu_count() // 2 # Número de workers. La mitad de los núcleos disponibles (heurística habitual)

## Carga de Datos

In [ ]:
from utils.load_california import load_housing_dataVAL_SIZE = 0.2X_train_full, X_test, y_train_full, y_test = load_housing_data()X_train, X_val, y_train, y_val = train_test_split(    X_train_full, y_train_full, test_size=VAL_SIZE, random_state=RANDOM_STATE)

Hemos separado un conjunto de validación. Por simplicidad, no realizaremos *validación cruzada* ya que es menos habitual en el *aprendizaje profundo*, dado que normalmente se usan mayores cantidades de datos para el entrenamiento, lo que aumenta el coste computacional. Sin embargo, en este caso el conjunto de datos no es tan grande y sería factible.

## Preprocesamiento de Datos

Se importan las funciones de preprocesamiento necesarias definidas en el módulo [`utils/housing_preprocessing.py`](utils/housing_preprocessing.py). El pipeline debe **ajustarse** (*fit*) *únicamente* con los datos de **entrenamiento** (`X_train`) para evitar la fuga de información. Luego se usa para **transformar** los demás conjuntos.

Tras el preprocesamiento, convertiremos los datos de arrays de NumPy a **Tensores de PyTorch**, que son la estructura de datos fundamental en PyTorch.

Para el número de clústeres en la similitud geoespacial, usamos el valor que dio los mejores resultados en [la búsqueda de hiperparámetros](e2e081_hyperparameters2.ipynb).

In [ ]:
from utils.housing_preprocessing import get_preprocessing_pipeline
preprocessing_pipeline = get_preprocessing_pipeline(n_clusters=76)
# Ajustar el pipeline SOLO con los datos de entrenamiento
preprocessing_pipeline.fit(X_train)
# Aplicar el pipeline para transformar los datos de entrada
X_train_processed = preprocessing_pipeline.transform(X_train)
X_val_processed = preprocessing_pipeline.transform(X_val)
X_test_processed = preprocessing_pipeline.transform(X_test)

La normalización de características es esencial para las Redes Neuronales porque llevar todos los atributos a escalas comparables evita que aquellos con magnitudes mayores dominen los cálculos de gradientes, lo que acelera la convergencia de los algoritmos de optimización y mejora la estabilidad numérica al evitar desbordamientos o gradientes "muertos" en las regiones planas de las funciones de activación. En este caso, el *pipeline* de preprocesamiento ya estandariza las variables de entrada.

Asimismo, normalizar la variable objetivo en tareas de regresión ayuda a que la función de pérdida opere en un rango razonable, facilita pasos de actualización más consistentes durante el aprendizaje, y puede mejorar tanto la velocidad de convergencia como la calidad final del modelo.

In [ ]:
from utils.housing_preprocessing import scale_targety_train_scaled_np, y_val_scaled_np, y_test_scaled_np, y_scaler = scale_target(y_train, y_val, y_test)

## Definición de la Métrica de Evaluación (y la Función de Pérdida)

La métrica de evaluación (RMSE) ya fue definida anteriormente. Por tanto, para los modelos de redes neuronales usaremos la función de pérdida `nn.MSELoss()`. Minimizar el MSE es equivalente a minimizar el RMSE. Usaremos el RMSE para evaluar el modelo con un valor más interpretable.

In [ ]:
criterion = nn.MSELoss()

## Línea Base: RandomForest

Comenzamos con un modelo RandomForest usando los mejores hiperparámetros encontrados anteriormente; con estos se obtuvo un RMSE de 41.604 USD mediante *validación cruzada*.

In [ ]:
# %%script false --no-raise-error # Para omitir una celda durante la ejecución
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=290,
    max_depth=87,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=RANDOM_STATE,
    n_jobs=num_workers
)
rf_model.fit(X_train_processed, y_train)
y_val_pred = rf_model.predict(X_val_processed)
val_rmse = root_mean_squared_error(y_val, y_val_pred)
print(f"RMSE de RandomForest: {val_rmse:.2f}")

Para el conjunto de validación definido en este caso, el modelo RandomForest alcanza un RMSE de **44.711 USD. Este será nuestro valor de referencia** para la comparación con los modelos de redes neuronales.

## Comparación con Modelos de Redes Neuronales

### Preparación de Datos

In [ ]:
# Convertir los datos de NumPy a Tensores de PyTorch y moverlos a la GPU si está disponible
X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32, device=device)
X_val_tensor = torch.tensor(X_val_processed, dtype=torch.float32, device=device)
X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32, device=device)
# Determinar el número de características de entrada para la red
n_features = X_train_tensor.shape[1]
print(f"Número de características de entrada: {n_features}")
y_train_tensor = torch.tensor(y_train_scaled_np, dtype=torch.float32, device=device)
y_val_tensor = torch.tensor(y_val_scaled_np, dtype=torch.float32, device=device)
y_test_tensor = torch.tensor(y_test_scaled_np, dtype=torch.float32, device=device)

Podemos mover el tensor a la GPU de antemano cuando la VRAM es suficiente, como en este conjunto de datos de "juguete". Sin embargo, no es el enfoque estándar para evitar errores de memoria insuficiente. Cada lote (*batch*) se moverá a la GPU más adelante durante el entrenamiento.

Usamos la clase `TensorDataset` para crear un conjunto de datos a partir de los tensores de características y etiquetas. Luego usamos la clase `DataLoader` para crear un `DataLoader` a partir del conjunto de datos. El `DataLoader` nos permite iterar sobre el conjunto de datos en *lotes* (*batches*), mezclando los datos en cada *época*.

In [ ]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)val_dataset = TensorDataset(X_val_tensor, y_val_tensor)test_dataset = TensorDataset(X_test_tensor, y_test_tensor)batch_size = 512train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

## Modelo 1: MLP Simple

In [ ]:
class SimpleMLP(nn.Module):    def __init__(self, n_features):        super().__init__()        self.layer1 = nn.Linear(n_features, 128)        self.relu1 = nn.ReLU()        self.layer2 = nn.Linear(128, 64)        self.relu2 = nn.ReLU()        self.layer3 = nn.Linear(64, 32)        self.relu3 = nn.ReLU()        self.output_layer = nn.Linear(32, 1)    def forward(self, x):        x = self.relu1(self.layer1(x))        x = self.relu2(self.layer2(x))        x = self.relu3(self.layer3(x))        x = self.output_layer(x)        return x    model1 = SimpleMLP(n_features).to(device)optimizer = optim.Adam(model1.parameters(), lr=0.001)

In [ ]:
from utils.training_utils import train_model_detailed, plot_metrics

num_epochs = 100
metrics1 = train_model_detailed(
    model=model1, 
    train_loader=train_loader, 
    val_loader=val_loader, 
    criterion=criterion, 
    optimizer=optimizer, 
    y_scaler=y_scaler, 
    num_epochs=num_epochs, 
    device=device,
    print_every=10 # Imprimir métricas cada 10 épocas
)
plot_metrics(metrics1)

## Modelo 2: Red Neuronal con Dropout

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class Model2(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(64, 1)  # regresión de un solo valor
        )

    def forward(self, x):
        return self.net(x)
    
model2 = Model2(n_features).to(device)
optimizer = optim.Adam(model2.parameters(), lr=0.001)

In [ ]:
metrics2 = train_model_detailed(
    model=model2, 
    train_loader=train_loader, 
    val_loader=val_loader, 
    criterion=criterion, 
    optimizer=optimizer, 
    y_scaler=y_scaler, 
    num_epochs=num_epochs, 
    device=device,
    print_every=10 # Imprimir métricas cada 10 épocas
)
plot_metrics(metrics1, metrics2)

Se observa que los nuevos modelos no mejoran el RMSE de referencia.

Sin embargo, cabe señalar que el modelo RandomForest fue sometido a un ajuste de hiperparámetros con *validación cruzada*, mientras que el modelo de red neuronal no ha sido ajustado.